In [ ]:
!pip install -q speechbrain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.1/782.1 kB 32.1 MB/s eta 0:00:00


In [ ]:
import os

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip install -q speechbrain jiwer==3.0.3 librosa soundfile transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.6 MB/s eta 0:00:00


In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/IJEEE_VC_data"
)

TARGET_REFERENCE = (
    PROJECT_DIR /
    "selected_hybrid_data/kathleen_reference.wav"
)

HYBRID_DIR = (
    PROJECT_DIR /
    "hybrid_tau_experiment"
)

HYBRID_METRICS_PATH = (
    HYBRID_DIR /
    "hybrid_tau_metrics.csv"
)

hybrid_outputs = list(
    HYBRID_DIR.rglob("vc_*.wav")
)

print(
    "Target reference exists:",
    TARGET_REFERENCE.exists()
)

print(
    "Metrics file exists:",
    HYBRID_METRICS_PATH.exists()
)

print(
    "Hybrid outputs:",
    len(hybrid_outputs)
)

Target reference exists: True
Metrics file exists: True
Hybrid outputs: 15


In [ ]:
import os

# Disable TensorFlow before importing Transformers
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q speechbrain jiwer==3.0.3 librosa soundfile transformers accelerate

In [ ]:
import re
import warnings

import librosa
import pandas as pd
import torch
import torch.nn.functional as F

from pathlib import Path
from jiwer import wer

from speechbrain.inference.speaker import (
    EncoderClassifier
)

from transformers import (
    AutoProcessor,
    AutoModelForSpeechSeq2Seq
)

warnings.filterwarnings("ignore")


# ============================================================
# 1. Paths
# ============================================================
PROJECT_DIR = Path(
    "/content/drive/MyDrive/IJEEE_VC_data"
)

TARGET_REFERENCE = (
    PROJECT_DIR /
    "selected_hybrid_data/kathleen_reference.wav"
)

HYBRID_DIR = (
    PROJECT_DIR /
    "hybrid_tau_experiment"
)

HYBRID_METRICS_PATH = (
    HYBRID_DIR /
    "hybrid_tau_metrics.csv"
)

assert TARGET_REFERENCE.exists(), (
    f"❌ Target reference missing: {TARGET_REFERENCE}"
)

assert HYBRID_METRICS_PATH.exists(), (
    f"❌ Metrics file missing: {HYBRID_METRICS_PATH}"
)


# ============================================================
# 2. Original transcripts
# ============================================================
REFERENCE_TRANSCRIPTS = {
    1: (
        "The monkey was planning to throw the net "
        "like the fishermen did."
    ),
    2: (
        "Gaze out to sea, picnic on benches and "
        "watch fishermen hook herring."
    ),
    3: (
        "We all have our own methods of dealing "
        "with money, but here are a couple of "
        "saving tricks that will grow your money."
    )
}


# ============================================================
# 3. Helper functions
# ============================================================
def normalize_text(text):
    text = text.lower()

    text = re.sub(
        r"[^a-z0-9'\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def load_audio_16k(audio_path):
    audio, _ = librosa.load(
        str(audio_path),
        sr=16000,
        mono=True
    )

    waveform = torch.from_numpy(
        audio
    ).float().unsqueeze(0)

    return waveform


# ============================================================
# 4. Read experiment metrics
# ============================================================
hybrid_metrics_df = pd.read_csv(
    HYBRID_METRICS_PATH
)

print("=" * 70)
print("EVALUATION SET CHECK")
print("=" * 70)

print(
    "Number of outputs:",
    len(hybrid_metrics_df)
)

print(
    "Tau values:",
    sorted(
        hybrid_metrics_df["tau"]
        .unique()
        .tolist()
    )
)

print(
    "Samples:",
    sorted(
        hybrid_metrics_df["sample"]
        .unique()
        .tolist()
    )
)

assert len(hybrid_metrics_df) == 15, (
    "❌ Expected 15 hybrid outputs"
)


# ============================================================
# 5. Load ECAPA-TDNN on CPU
# ============================================================
print("\nLoading ECAPA-TDNN on CPU...")

speaker_encoder = (
    EncoderClassifier.from_hparams(
        source=(
            "speechbrain/"
            "spkrec-ecapa-voxceleb"
        ),
        savedir=(
            "/content/pretrained_models/"
            "spkrec-ecapa-voxceleb"
        ),
        run_opts={
            "device": "cpu"
        }
    )
)

target_waveform = load_audio_16k(
    TARGET_REFERENCE
)

with torch.inference_mode():
    target_embedding = (
        speaker_encoder.encode_batch(
            target_waveform
        )
    )

print("✅ ECAPA-TDNN loaded")
print(
    "Target embedding shape:",
    tuple(target_embedding.shape)
)


# ============================================================
# 6. Load Whisper-small on CPU
# ============================================================
print("\nLoading Whisper-small on CPU...")

WHISPER_MODEL = "openai/whisper-small"

whisper_processor = (
    AutoProcessor.from_pretrained(
        WHISPER_MODEL
    )
)

whisper_asr = (
    AutoModelForSpeechSeq2Seq
    .from_pretrained(
        WHISPER_MODEL,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    )
    .to("cpu")
    .eval()
)

print("✅ Whisper-small loaded")


def transcribe_with_whisper(audio_path):
    audio, _ = librosa.load(
        str(audio_path),
        sr=16000,
        mono=True
    )

    inputs = whisper_processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
        return_attention_mask=True
    )

    generate_arguments = {
        "input_features": (
            inputs.input_features
        ),
        "language": "english",
        "task": "transcribe",
        "do_sample": False,
        "num_beams": 1,
        "max_new_tokens": 128
    }

    if hasattr(inputs, "attention_mask"):
        generate_arguments[
            "attention_mask"
        ] = inputs.attention_mask

    with torch.inference_mode():
        predicted_ids = (
            whisper_asr.generate(
                **generate_arguments
            )
        )

    transcription = (
        whisper_processor.batch_decode(
            predicted_ids,
            skip_special_tokens=True
        )[0]
    )

    return transcription.strip()


# ============================================================
# 7. Evaluate all outputs
# ============================================================
evaluation_records = []

total_outputs = len(
    hybrid_metrics_df
)

for position, (_, row) in enumerate(
    hybrid_metrics_df.iterrows(),
    start=1
):
    tau = float(row["tau"])
    sample = int(row["sample"])

    output_path = Path(
        row["drive_hybrid_output"]
    )

    assert output_path.exists(), (
        f"❌ Output missing: {output_path}"
    )

    print("\n" + "-" * 70)
    print(
        f"Evaluating {position}/{total_outputs}"
    )
    print(
        f"Tau={tau} | Sample={sample}"
    )

    # --------------------------------------------------------
    # Speaker similarity
    # --------------------------------------------------------
    output_waveform = load_audio_16k(
        output_path
    )

    with torch.inference_mode():
        output_embedding = (
            speaker_encoder.encode_batch(
                output_waveform
            )
        )

    similarity = F.cosine_similarity(
        output_embedding.reshape(1, -1),
        target_embedding.reshape(1, -1),
        dim=1
    ).item()

    # --------------------------------------------------------
    # Whisper transcription
    # --------------------------------------------------------
    hypothesis = (
        transcribe_with_whisper(
            output_path
        )
    )

    reference = (
        REFERENCE_TRANSCRIPTS[sample]
    )

    normalized_reference = (
        normalize_text(reference)
    )

    normalized_hypothesis = (
        normalize_text(hypothesis)
    )

    sample_wer = wer(
        normalized_reference,
        normalized_hypothesis
    )

    evaluation_records.append({
        "tau": tau,
        "sample": sample,
        "speaker_similarity": similarity,
        "wer": sample_wer,
        "reference_text": reference,
        "whisper_transcription": hypothesis,
        "normalized_reference": (
            normalized_reference
        ),
        "normalized_hypothesis": (
            normalized_hypothesis
        ),
        "evaluated_output": str(
            output_path
        )
    })

    print(
        "Speaker similarity:",
        round(similarity, 4)
    )

    print(
        "WER:",
        round(sample_wer, 4)
    )

    print(
        "Reference:",
        reference
    )

    print(
        "Whisper:",
        hypothesis
    )

    # Save progress after every output
    progress_df = pd.DataFrame(
        evaluation_records
    )

    progress_df.to_csv(
        HYBRID_DIR /
        "objective_evaluation_progress.csv",
        index=False
    )


# ============================================================
# 8. Merge timing and objective metrics
# ============================================================
objective_df = pd.DataFrame(
    evaluation_records
)

complete_df = hybrid_metrics_df.merge(
    objective_df,
    on=[
        "tau",
        "sample"
    ],
    how="left"
)

COMPLETE_RESULTS_PATH = (
    HYBRID_DIR /
    "hybrid_tau_complete_evaluation.csv"
)

complete_df.to_csv(
    COMPLETE_RESULTS_PATH,
    index=False
)


# ============================================================
# 9. Summary for each tau
# ============================================================
objective_summary = (
    complete_df
    .groupby("tau")
    .agg(
        mean_similarity=(
            "speaker_similarity",
            "mean"
        ),
        std_similarity=(
            "speaker_similarity",
            "std"
        ),
        mean_wer=(
            "wer",
            "mean"
        ),
        std_wer=(
            "wer",
            "std"
        ),
        mean_hybrid_latency=(
            "hybrid_latency_seconds",
            "mean"
        ),
        std_hybrid_latency=(
            "hybrid_latency_seconds",
            "std"
        ),
        mean_hybrid_rtf=(
            "hybrid_rtf",
            "mean"
        ),
        std_hybrid_rtf=(
            "hybrid_rtf",
            "std"
        )
    )
    .reset_index()
)

SUMMARY_PATH = (
    HYBRID_DIR /
    "hybrid_tau_objective_summary.csv"
)

objective_summary.to_csv(
    SUMMARY_PATH,
    index=False
)


# ============================================================
# 10. Print final table
# ============================================================
print("\n" + "=" * 85)
print("COMPLETE HYBRID TAU EVALUATION")
print("=" * 85)

print(
    objective_summary.round(4).to_string(
        index=False
    )
)

print("\n✅ Complete results saved:")
print(COMPLETE_RESULTS_PATH)

print("\n✅ Summary saved:")
print(SUMMARY_PATH)

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


EVALUATION SET CHECK
Number of outputs: 15
Tau values: [0.1, 0.3, 0.5, 0.7, 0.9]
Samples: [1, 2, 3]

Loading ECAPA-TDNN on CPU...


hyperparams.yaml:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


embedding_model.ckpt: reconstructing file:   0%|          |  0.00B / 83.3MB            

embedding_model.ckpt: downloading bytes:           |  0.00B            

... (50 outputs left out)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Speaker similarity: 0.7286
WER: 0.1667
Reference: Gaze out to sea, picnic on benches and watch fishermen hook herring.
Whisper: Gaze out to see, picnic on benches and watch fishermen hoke herring.

----------------------------------------------------------------------
Evaluating 15/15
Tau=0.9 | Sample=3


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Speaker similarity: 0.7811
WER: 0.0
Reference: We all have our own methods of dealing with money, but here are a couple of saving tricks that will grow your money.
Whisper: We all have our own methods of dealing with money, but here are a couple of saving tricks that will grow your money.

COMPLETE HYBRID TAU EVALUATION
 tau  mean_similarity  std_similarity  mean_wer  std_wer  mean_hybrid_latency  std_hybrid_latency  mean_hybrid_rtf  std_hybrid_rtf
 0.1           0.7438          0.0411    0.0556   0.0481               2.8476              0.5040           0.4836          0.0451
 0.3           0.7433          0.0320    0.1111   0.1273               3.0356              0.9282           0.5066          0.0728
 0.5           0.7480          0.0197    0.0556   0.0481               2.7090              0.1655           0.4698          0.1020
 0.7           0.7469          0.0384    0.0833   0.0833               2.3614              0.2201           0.4033          0.0284
 0.9           0.7495  

In [ ]:
# ============================================================
# CPU EVALUATION — STEP 1: CHECK SAVED OUTPUTS
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import glob
import torch

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

EXPERIMENT_DIR = os.path.join(
    BASE_DIR,
    "hybrid_seed_steps_experiment"
)

TARGET_PATH = os.path.join(
    BASE_DIR,
    "selected_hybrid_data",
    "kathleen_reference.wav"
)

HYBRID_OUTPUTS = sorted(
    glob.glob(
        os.path.join(
            EXPERIMENT_DIR,
            "hybrid_sample_*_tau_0.5_steps_*.wav"
        )
    )
)

print("=" * 70)
print("SEED-STEPS CPU EVALUATION CHECK")
print("=" * 70)

print("CUDA available:", torch.cuda.is_available())
print("Kathleen reference exists:", os.path.isfile(TARGET_PATH))
print("Hybrid Seed-step outputs:", len(HYBRID_OUTPUTS))

for path in HYBRID_OUTPUTS:
    print(" -", os.path.basename(path))

assert os.path.isfile(TARGET_PATH), (
    "❌ Kathleen reference is missing"
)

assert len(HYBRID_OUTPUTS) == 15, (
    f"❌ Expected 15 outputs, found {len(HYBRID_OUTPUTS)}"
)

print("\n✅ All 15 outputs are ready for CPU evaluation")

Mounted at /content/drive
SEED-STEPS CPU EVALUATION CHECK
CUDA available: False
Kathleen reference exists: True
Hybrid Seed-step outputs: 15
 - hybrid_sample_01_tau_0.5_steps_05.wav
 - hybrid_sample_01_tau_0.5_steps_10.wav
 - hybrid_sample_01_tau_0.5_steps_20.wav
 - hybrid_sample_01_tau_0.5_steps_30.wav
 - hybrid_sample_01_tau_0.5_steps_50.wav
 - hybrid_sample_02_tau_0.5_steps_05.wav
 - hybrid_sample_02_tau_0.5_steps_10.wav
 - hybrid_sample_02_tau_0.5_steps_20.wav
 - hybrid_sample_02_tau_0.5_steps_30.wav
 - hybrid_sample_02_tau_0.5_steps_50.wav
 - hybrid_sample_03_tau_0.5_steps_05.wav
 - hybrid_sample_03_tau_0.5_steps_10.wav
 - hybrid_sample_03_tau_0.5_steps_20.wav
 - hybrid_sample_03_tau_0.5_steps_30.wav
 - hybrid_sample_03_tau_0.5_steps_50.wav

✅ All 15 outputs are ready for CPU evaluation


In [ ]:
# ============================================================
# CPU EVALUATION — STEP 2: INSTALL DEPENDENCIES
# ============================================================

import os
import sys
import subprocess

# منع Transformers من محاولة تشغيل TensorFlow
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

evaluation_packages = [
    "numpy==1.26.4",
    "scipy==1.13.1",
    "librosa==0.10.2",
    "soundfile==0.12.1",
    "transformers==4.46.3",
    "huggingface-hub>=0.28.1",
    "speechbrain==1.0.3",
    "jiwer==3.0.5"
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q"
    ] + evaluation_packages,
    check=True
)

print("=" * 70)
print("CPU EVALUATION DEPENDENCIES INSTALLED")
print("=" * 70)

print("✅ SpeechBrain installed")
print("✅ JiWER installed")
print("✅ Transformers 4.46.3 installed")
print("✅ Librosa 0.10.2 installed")

print("\n⚠️ الآن اعملي Runtime → Restart session")

CPU EVALUATION DEPENDENCIES INSTALLED
✅ SpeechBrain installed
✅ JiWER installed
✅ Transformers 4.46.3 installed
✅ Librosa 0.10.2 installed

⚠️ الآن اعملي Runtime → Restart session


In [ ]:
# ============================================================
# CPU EVALUATION — STEP 3 FIXED
# TorchAudio 2.11 + SpeechBrain compatibility
# ============================================================

import os
import sys

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

from google.colab import drive
drive.mount("/content/drive")

import glob
import torch
import torchaudio
import numpy as np
import librosa
import transformers

# ------------------------------------------------------------
# Compatibility patch for TorchAudio >= 2.9
# ------------------------------------------------------------

if not hasattr(torchaudio, "list_audio_backends"):
    torchaudio.list_audio_backends = lambda: ["soundfile"]

# إزالة أي استيراد جزئي فاشل سابق لـSpeechBrain
for module_name in list(sys.modules):
    if (
        module_name == "speechbrain"
        or module_name.startswith("speechbrain.")
    ):
        del sys.modules[module_name]

import speechbrain
import jiwer

from speechbrain.inference.speaker import SpeakerRecognition
from transformers import (
    AutoProcessor,
    AutoModelForSpeechSeq2Seq
)

# ------------------------------------------------------------
# Experiment paths
# ------------------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

EXPERIMENT_DIR = os.path.join(
    BASE_DIR,
    "hybrid_seed_steps_experiment"
)

TARGET_PATH = os.path.join(
    BASE_DIR,
    "selected_hybrid_data",
    "kathleen_reference.wav"
)

HYBRID_OUTPUTS = sorted(
    glob.glob(
        os.path.join(
            EXPERIMENT_DIR,
            "hybrid_sample_*_tau_0.5_steps_*.wav"
        )
    )
)

assert not torch.cuda.is_available(), (
    "❌ هذه الجلسة يجب أن تكون CPU"
)

assert os.path.isfile(TARGET_PATH)
assert len(HYBRID_OUTPUTS) == 15

print("=" * 70)
print("CPU EVALUATION ENVIRONMENT")
print("=" * 70)

print("Torch:", torch.__version__)
print("TorchAudio:", torchaudio.__version__)
print("NumPy:", np.__version__)
print("Librosa:", librosa.__version__)
print("Transformers:", transformers.__version__)
print("SpeechBrain:", speechbrain.__version__)
print("CUDA available:", torch.cuda.is_available())
print("TensorFlow enabled:", transformers.utils.is_tf_available())
print("Evaluation files:", len(HYBRID_OUTPUTS))

# ------------------------------------------------------------
# Load SpeechBrain ECAPA-TDNN
# ------------------------------------------------------------

print("\nLoading SpeechBrain ECAPA-TDNN...")

speaker_model = SpeakerRecognition.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/content/evaluation_models/ecapa_voxceleb",
    run_opts={"device": "cpu"}
)

print("✅ SpeechBrain ECAPA-TDNN loaded")

# ------------------------------------------------------------
# Load Whisper-small
# ------------------------------------------------------------

WHISPER_MODEL_NAME = "openai/whisper-small"

print("\nLoading Whisper-small...")
print("The first load may download approximately 1 GB.\n")

whisper_processor = AutoProcessor.from_pretrained(
    WHISPER_MODEL_NAME
)

whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    WHISPER_MODEL_NAME,
    torch_dtype=torch.float32,
    use_safetensors=True
)

whisper_model.to("cpu")
whisper_model.eval()

print("\n" + "=" * 70)
print("✅ CPU EVALUATION MODELS READY")
print("=" * 70)

print("Speaker model: SpeechBrain ECAPA-TDNN")
print("ASR model: Whisper-small")
print("Device: CPU")
print("Files awaiting evaluation:", len(HYBRID_OUTPUTS))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _speechbrain_save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _speechbrain_load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _recover
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


CPU EVALUATION ENVIRONMENT
Torch: 2.11.0+cpu
TorchAudio: 2.11.0+cpu
NumPy: 1.26.4
Librosa: 0.10.2
Transformers: 4.46.3
SpeechBrain: 1.0.3
CUDA available: False
TensorFlow enabled: False
Evaluation files: 15

Loading SpeechBrain ECAPA-TDNN...


hyperparams.yaml: 0.00B [00:00, ?B/s]

DEBUG:speechbrain.utils.fetching:Fetch: Local file found, creating symlink '/root/.cache/huggingface/hub/models--speechbrain--spkrec-ecapa-voxceleb/snapshots/0f99f2d0ebe89ac095bcc5903c4dd8f72b367286/hyperparams.yaml' -> '/content/evaluation_models/ecapa_voxceleb/hyperparams.yaml'
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _load
DEBUG:speechbrain.utils.checkpoints:Registered parameter transfer hook for _load
/usr/local/lib/python3.13/dist-packages/speechbrain/utils/autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook

embedding_model.ckpt:   0%|          | 0.00/83.3M [00:00<?, ?B/s]

... (17 outputs left out)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]


✅ CPU EVALUATION MODELS READY
Speaker model: SpeechBrain ECAPA-TDNN
ASR model: Whisper-small
Device: CPU
Files awaiting evaluation: 15


In [ ]:
# ============================================================
# CPU EVALUATION — STEP 4: COMPLETE QUALITY EVALUATION
# Speaker similarity + WER + timing merge
# ============================================================

import os
import re
import glob
import string

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn.functional as F

from jiwer import wer

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

EXPERIMENT_DIR = os.path.join(
    BASE_DIR,
    "hybrid_seed_steps_experiment"
)

TARGET_PATH = os.path.join(
    BASE_DIR,
    "selected_hybrid_data",
    "kathleen_reference.wav"
)

TIMING_CSV_PATH = os.path.join(
    EXPERIMENT_DIR,
    "seed_steps_sample_metrics.csv"
)

PARTIAL_CSV_PATH = os.path.join(
    EXPERIMENT_DIR,
    "seed_steps_quality_evaluation_partial.csv"
)

COMPLETE_CSV_PATH = os.path.join(
    EXPERIMENT_DIR,
    "seed_steps_complete_evaluation.csv"
)

SUMMARY_CSV_PATH = os.path.join(
    EXPERIMENT_DIR,
    "seed_steps_objective_summary.csv"
)

HYBRID_OUTPUTS = sorted(
    glob.glob(
        os.path.join(
            EXPERIMENT_DIR,
            "hybrid_sample_*_tau_0.5_steps_*.wav"
        )
    )
)

assert len(HYBRID_OUTPUTS) == 15
assert os.path.isfile(TARGET_PATH)
assert os.path.isfile(TIMING_CSV_PATH)
assert "speaker_model" in globals()
assert "whisper_model" in globals()
assert "whisper_processor" in globals()

# ------------------------------------------------------------
# Fixed reference transcripts
# ------------------------------------------------------------

REFERENCE_TRANSCRIPTS = {
    1: (
        "The monkey was planning to throw the net "
        "like the fishermen did."
    ),
    2: (
        "Gaze out to sea, picnic on benches and "
        "watch fishermen hook herring."
    ),
    3: (
        "We all have our own methods of dealing with money, "
        "but here are a couple of saving tricks that will "
        "grow your money."
    )
}

# ------------------------------------------------------------
# Utility functions
# ------------------------------------------------------------

def normalize_text(text):
    """Lowercase, remove punctuation and normalize spaces."""

    text = text.lower().strip()

    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    text = " ".join(text.split())

    return text


def load_audio_16k(audio_path):
    """Load audio as mono at 16 kHz using librosa."""

    waveform, _ = librosa.load(
        audio_path,
        sr=16000,
        mono=True
    )

    waveform = np.asarray(
        waveform,
        dtype=np.float32
    )

    return waveform


def extract_speaker_embedding(audio_path):
    """Extract SpeechBrain ECAPA-TDNN speaker embedding."""

    waveform = load_audio_16k(audio_path)

    waveform_tensor = torch.from_numpy(
        waveform
    ).unsqueeze(0)

    relative_length = torch.tensor(
        [1.0],
        dtype=torch.float32
    )

    with torch.no_grad():
        embedding = speaker_model.encode_batch(
            waveform_tensor,
            relative_length,
            normalize=False
        )

    return embedding.reshape(1, -1)


def transcribe_with_whisper(audio_path):
    """Transcribe one audio file with Whisper-small."""

    waveform = load_audio_16k(audio_path)

    whisper_inputs = whisper_processor(
        waveform,
        sampling_rate=16000,
        return_tensors="pt",
        return_attention_mask=True
    )

    generation_inputs = {
        "input_features": whisper_inputs.input_features
    }

    if hasattr(whisper_inputs, "attention_mask"):
        generation_inputs["attention_mask"] = (
            whisper_inputs.attention_mask
        )

    with torch.no_grad():
        generated_ids = whisper_model.generate(
            **generation_inputs,
            language="en",
            task="transcribe",
            max_new_tokens=128
        )

    transcription = whisper_processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return transcription.strip()


def parse_output_filename(audio_path):
    """Extract sample and diffusion steps from filename."""

    filename = os.path.basename(audio_path)

    match = re.match(
        r"hybrid_sample_(\d+)_tau_0\.5_steps_(\d+)\.wav",
        filename
    )

    if not match:
        raise ValueError(
            f"Unexpected filename: {filename}"
        )

    sample_number = int(match.group(1))
    diffusion_steps = int(match.group(2))

    return sample_number, diffusion_steps


# ------------------------------------------------------------
# Extract fixed target embedding once
# ------------------------------------------------------------

print("Extracting fixed Kathleen target embedding...")

target_embedding = extract_speaker_embedding(
    TARGET_PATH
)

print("✅ Target embedding extracted")

# ------------------------------------------------------------
# Resume support
# ------------------------------------------------------------

if os.path.isfile(PARTIAL_CSV_PATH):

    existing_df = pd.read_csv(PARTIAL_CSV_PATH)
    evaluation_records = existing_df.to_dict("records")

    completed_cases = {
        (
            int(row["sample"]),
            int(row["diffusion_steps"])
        )
        for _, row in existing_df.iterrows()
    }

    print(
        f"Resuming from {len(completed_cases)} "
        "previously evaluated files."
    )

else:

    evaluation_records = []
    completed_cases = set()

# Sort primarily by steps, then by sample
ordered_outputs = sorted(
    HYBRID_OUTPUTS,
    key=lambda path: (
        parse_output_filename(path)[1],
        parse_output_filename(path)[0]
    )
)

total_files = len(ordered_outputs)

# ------------------------------------------------------------
# Main evaluation
# ------------------------------------------------------------

for file_index, audio_path in enumerate(
    ordered_outputs,
    start=1
):

    sample_number, diffusion_steps = (
        parse_output_filename(audio_path)
    )

    case_key = (
        sample_number,
        diffusion_steps
    )

    if case_key in completed_cases:
        print(
            f"Skipping completed case: "
            f"steps={diffusion_steps}, "
            f"sample={sample_number}"
        )
        continue

    reference_text = REFERENCE_TRANSCRIPTS[
        sample_number
    ]

    print("\n" + "-" * 75)
    print(f"Evaluating {file_index}/{total_files}")
    print(
        f"Steps={diffusion_steps} | "
        f"Sample={sample_number}"
    )

    # Speaker similarity
    output_embedding = extract_speaker_embedding(
        audio_path
    )

    similarity = F.cosine_similarity(
        target_embedding,
        output_embedding,
        dim=1
    ).item()

    # Whisper transcription and WER
    whisper_text = transcribe_with_whisper(
        audio_path
    )

    normalized_reference = normalize_text(
        reference_text
    )

    normalized_hypothesis = normalize_text(
        whisper_text
    )

    if normalized_hypothesis:
        word_error_rate = wer(
            normalized_reference,
            normalized_hypothesis
        )
    else:
        word_error_rate = 1.0

    evaluation_records.append({
        "tau": 0.5,
        "diffusion_steps": diffusion_steps,
        "sample": sample_number,
        "speaker_similarity": similarity,
        "wer": word_error_rate,
        "reference_transcript": reference_text,
        "whisper_transcript": whisper_text,
        "output_path": audio_path
    })

    # Save after every evaluated file
    partial_df = pd.DataFrame(
        evaluation_records
    ).sort_values(
        ["diffusion_steps", "sample"]
    )

    partial_df.to_csv(
        PARTIAL_CSV_PATH,
        index=False
    )

    completed_cases.add(case_key)

    print("Speaker similarity:", round(similarity, 4))
    print("WER:", round(word_error_rate, 4))
    print("Reference:", reference_text)
    print("Whisper:", whisper_text)

# ------------------------------------------------------------
# Merge quality and timing metrics
# ------------------------------------------------------------

quality_df = pd.DataFrame(
    evaluation_records
).sort_values(
    ["diffusion_steps", "sample"]
)

timing_df = pd.read_csv(
    TIMING_CSV_PATH
)

complete_df = quality_df.merge(
    timing_df[
        [
            "diffusion_steps",
            "sample",
            "duration_seconds",
            "openvoice_latency_seconds",
            "mean_seed_latency_seconds",
            "std_seed_latency_seconds",
            "mean_seed_rtf",
            "mean_hybrid_latency_seconds",
            "mean_hybrid_rtf"
        ]
    ],
    on=["diffusion_steps", "sample"],
    how="left"
)

complete_df.to_csv(
    COMPLETE_CSV_PATH,
    index=False
)

# ------------------------------------------------------------
# Final objective summary
# ------------------------------------------------------------

summary_df = (
    complete_df
    .groupby("diffusion_steps", as_index=False)
    .agg(
        mean_similarity=(
            "speaker_similarity",
            "mean"
        ),
        std_similarity=(
            "speaker_similarity",
            "std"
        ),
        mean_wer=(
            "wer",
            "mean"
        ),
        std_wer=(
            "wer",
            "std"
        ),
        mean_seed_latency=(
            "mean_seed_latency_seconds",
            "mean"
        ),
        mean_seed_rtf=(
            "mean_seed_rtf",
            "mean"
        ),
        mean_hybrid_latency=(
            "mean_hybrid_latency_seconds",
            "mean"
        ),
        mean_hybrid_rtf=(
            "mean_hybrid_rtf",
            "mean"
        )
    )
)

summary_df.insert(0, "tau", 0.5)

summary_df.to_csv(
    SUMMARY_CSV_PATH,
    index=False
)

print("\n" + "=" * 90)
print("COMPLETE SEED DIFFUSION-STEPS EVALUATION")
print("=" * 90)

print(
    summary_df.round(4).to_string(index=False)
)

print("\n✅ Complete evaluation:", COMPLETE_CSV_PATH)
print("✅ Objective summary:", SUMMARY_CSV_PATH)

Extracting fixed Kathleen target embedding...
✅ Target embedding extracted
Resuming from 15 previously evaluated files.
Skipping completed case: steps=5, sample=1
Skipping completed case: steps=5, sample=2
Skipping completed case: steps=5, sample=3
Skipping completed case: steps=10, sample=1
Skipping completed case: steps=10, sample=2
Skipping completed case: steps=10, sample=3
Skipping completed case: steps=20, sample=1
Skipping completed case: steps=20, sample=2
Skipping completed case: steps=20, sample=3
Skipping completed case: steps=30, sample=1
Skipping completed case: steps=30, sample=2
Skipping completed case: steps=30, sample=3
Skipping completed case: steps=50, sample=1
Skipping completed case: steps=50, sample=2
Skipping completed case: steps=50, sample=3

COMPLETE SEED DIFFUSION-STEPS EVALUATION
 tau  diffusion_steps  mean_similarity  std_similarity  mean_wer  std_wer  mean_seed_latency  mean_seed_rtf  mean_hybrid_latency  mean_hybrid_rtf
 0.5                5           0.7

In [ ]:
# ============================================================
# FINAL COMPARISON — CPU STEP 1: CHECK ALL 40 OUTPUTS
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import glob
import torch

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

COMPARISON_DIR = os.path.join(
    BASE_DIR,
    "final_hybrid_comparison"
)

OPENVOICE_FILES = sorted(
    glob.glob(
        os.path.join(
            COMPARISON_DIR,
            "openvoice_outputs",
            "tau_0.5",
            "openvoice_sample_*_tau_0.5.wav"
        )
    )
)

SEED_ONLY_FILES = sorted(
    glob.glob(
        os.path.join(
            COMPARISON_DIR,
            "seed_based_outputs",
            "seed_only",
            "seed_only_sample_*_steps_10.wav"
        )
    )
)

DEFAULT_HYBRID_FILES = sorted(
    glob.glob(
        os.path.join(
            COMPARISON_DIR,
            "seed_based_outputs",
            "default_hybrid",
            "default_hybrid_sample_*_tau_0.3_steps_20.wav"
        )
    )
)

OPTIMIZED_HYBRID_FILES = sorted(
    glob.glob(
        os.path.join(
            COMPARISON_DIR,
            "seed_based_outputs",
            "optimized_hybrid",
            "optimized_hybrid_sample_*_tau_0.5_steps_10.wav"
        )
    )
)

TIMING_METRICS_PATH = os.path.join(
    COMPARISON_DIR,
    "four_system_timing_sample_metrics.csv"
)

METADATA_PATH = os.path.join(
    BASE_DIR,
    "final_holdout_data",
    "holdout_metadata.csv"
)

TARGET_PATH = os.path.join(
    BASE_DIR,
    "selected_hybrid_data",
    "kathleen_reference.wav"
)

print("=" * 75)
print("FINAL FOUR-SYSTEM CPU EVALUATION CHECK")
print("=" * 75)

print("CUDA available:", torch.cuda.is_available())
print("OpenVoice Only:", len(OPENVOICE_FILES))
print("Seed-VC Only:", len(SEED_ONLY_FILES))
print("Default Hybrid:", len(DEFAULT_HYBRID_FILES))
print("Optimized Hybrid:", len(OPTIMIZED_HYBRID_FILES))

print("\nTiming metrics exist:", os.path.isfile(TIMING_METRICS_PATH))
print("Metadata exists:", os.path.isfile(METADATA_PATH))
print("Kathleen reference exists:", os.path.isfile(TARGET_PATH))

assert not torch.cuda.is_available(), (
    "❌ هذه الخلية يجب تشغيلها على CPU"
)

assert len(OPENVOICE_FILES) == 10
assert len(SEED_ONLY_FILES) == 10
assert len(DEFAULT_HYBRID_FILES) == 10
assert len(OPTIMIZED_HYBRID_FILES) == 10

assert os.path.isfile(TIMING_METRICS_PATH)
assert os.path.isfile(METADATA_PATH)
assert os.path.isfile(TARGET_PATH)

print("\n✅ All 40 outputs are ready for objective evaluation")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FINAL FOUR-SYSTEM CPU EVALUATION CHECK
CUDA available: False
OpenVoice Only: 10
Seed-VC Only: 10
Default Hybrid: 10
Optimized Hybrid: 10

Timing metrics exist: True
Metadata exists: True
Kathleen reference exists: True

✅ All 40 outputs are ready for objective evaluation


In [ ]:
# ============================================================
# FINAL COMPARISON — CPU STEP 2: INSTALL EVALUATION PACKAGES
# ============================================================

import os
import sys
import subprocess

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

evaluation_packages = [
    "numpy==1.26.4",
    "scipy==1.13.1",
    "librosa==0.10.2",
    "soundfile==0.12.1",
    "transformers==4.46.3",
    "huggingface-hub>=0.28.1",
    "speechbrain==1.0.3",
    "jiwer==3.0.5"
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q"
    ] + evaluation_packages,
    check=True
)

print("=" * 75)
print("FINAL EVALUATION DEPENDENCIES INSTALLED")
print("=" * 75)

print("✅ SpeechBrain 1.0.3")
print("✅ JiWER 3.0.5")
print("✅ Transformers 4.46.3")
print("✅ Librosa 0.10.2")

print("\n⚠️ الآن اعملي Runtime → Restart session")

FINAL EVALUATION DEPENDENCIES INSTALLED
✅ SpeechBrain 1.0.3
✅ JiWER 3.0.5
✅ Transformers 4.46.3
✅ Librosa 0.10.2

⚠️ الآن اعملي Runtime → Restart session


In [ ]:
# ============================================================
# FINAL FOUR-SYSTEM EVALUATION — LOAD CPU MODELS
# Run this cell AFTER restarting the CPU session
# ============================================================

import os
import sys
import glob
import logging

# منع Transformers من محاولة تحميل TensorFlow
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

from google.colab import drive
drive.mount("/content/drive")

import torch
import torchaudio

# إصلاح توافق SpeechBrain 1.0.3 مع TorchAudio الحديث
if not hasattr(torchaudio, "list_audio_backends"):
    torchaudio.list_audio_backends = lambda: ["soundfile"]

import numpy as np
import librosa
import transformers
import speechbrain
import jiwer

from speechbrain.inference.speaker import SpeakerRecognition
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

logging.getLogger("speechbrain").setLevel(logging.WARNING)

# ============================================================
# المسارات
# ============================================================

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

COMPARISON_DIR = os.path.join(
    BASE_DIR,
    "final_hybrid_comparison"
)

TARGET_PATH = os.path.join(
    BASE_DIR,
    "selected_hybrid_data",
    "kathleen_reference.wav"
)

METADATA_PATH = os.path.join(
    BASE_DIR,
    "final_holdout_data",
    "holdout_metadata.csv"
)

OPENVOICE_FILES = sorted(glob.glob(
    os.path.join(
        COMPARISON_DIR,
        "openvoice_outputs",
        "tau_0.5",
        "openvoice_sample_*_tau_0.5.wav"
    )
))

SEED_ONLY_FILES = sorted(glob.glob(
    os.path.join(
        COMPARISON_DIR,
        "seed_based_outputs",
        "seed_only",
        "seed_only_sample_*_steps_10.wav"
    )
))

DEFAULT_HYBRID_FILES = sorted(glob.glob(
    os.path.join(
        COMPARISON_DIR,
        "seed_based_outputs",
        "default_hybrid",
        "default_hybrid_sample_*_tau_0.3_steps_20.wav"
    )
))

OPTIMIZED_HYBRID_FILES = sorted(glob.glob(
    os.path.join(
        COMPARISON_DIR,
        "seed_based_outputs",
        "optimized_hybrid",
        "optimized_hybrid_sample_*_tau_0.5_steps_10.wav"
    )
))

print("=" * 75)
print("FINAL FOUR-SYSTEM CPU ENVIRONMENT CHECK")
print("=" * 75)

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("TorchAudio:", torchaudio.__version__)
print("SpeechBrain:", speechbrain.__version__)
print("Transformers:", transformers.__version__)
print("Librosa:", librosa.__version__)
print("CUDA available:", torch.cuda.is_available())

print("\nKathleen reference:", os.path.isfile(TARGET_PATH))
print("Holdout metadata:", os.path.isfile(METADATA_PATH))
print("OpenVoice Only:", len(OPENVOICE_FILES))
print("Seed-VC Only:", len(SEED_ONLY_FILES))
print("Default Hybrid:", len(DEFAULT_HYBRID_FILES))
print("Optimized Hybrid:", len(OPTIMIZED_HYBRID_FILES))

assert not torch.cuda.is_available(), "❌ الجلسة ليست CPU"
assert os.path.isfile(TARGET_PATH), "❌ Kathleen reference is missing"
assert os.path.isfile(METADATA_PATH), "❌ Holdout metadata is missing"
assert len(OPENVOICE_FILES) == 10, "❌ OpenVoice outputs are incomplete"
assert len(SEED_ONLY_FILES) == 10, "❌ Seed-VC outputs are incomplete"
assert len(DEFAULT_HYBRID_FILES) == 10, "❌ Default Hybrid outputs are incomplete"
assert len(OPTIMIZED_HYBRID_FILES) == 10, "❌ Optimized Hybrid outputs are incomplete"

# ============================================================
# تحميل نموذج تشابه المتحدث
# ============================================================

print("\nLoading SpeechBrain ECAPA-TDNN...")

speaker_model = SpeakerRecognition.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/content/evaluation_models/ecapa_voxceleb",
    run_opts={"device": "cpu"}
)

print("✅ SpeechBrain ECAPA-TDNN loaded")

# ============================================================
# تحميل Whisper-small لحساب WER
# ============================================================

print("\nLoading Whisper-small...")
print("The first load may download approximately 1 GB.\n")

whisper_processor = AutoProcessor.from_pretrained(
    "openai/whisper-small"
)

whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    "openai/whisper-small",
    torch_dtype=torch.float32,
    use_safetensors=True
)

whisper_model.to("cpu")
whisper_model.eval()

# إزالة تعارض forced_decoder_ids
whisper_model.config.forced_decoder_ids = None
whisper_model.generation_config.forced_decoder_ids = None

print("\n" + "=" * 75)
print("✅ CPU EVALUATION MODELS READY")
print("=" * 75)
print("Speaker similarity model: SpeechBrain ECAPA-TDNN")
print("Speech recognition model: Whisper-small")
print("Evaluation device: CPU")
print("Total outputs awaiting evaluation: 40")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _speechbrain_save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _speechbrain_load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _recover
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/evaluation_models/ecapa_voxceleb/hyperparams.yaml'
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _load
DEBUG:speechbrain.utils.checkpoints:Registered parameter transfer hook for _load


FINAL FOUR-SYSTEM CPU ENVIRONMENT CHECK
Python: 3.13.15
Torch: 2.11.0+cpu
TorchAudio: 2.11.0+cpu
SpeechBrain: 1.0.3
Transformers: 4.46.3
Librosa: 0.10.2
CUDA available: False

Kathleen reference: True
Holdout metadata: True
OpenVoice Only: 10
Seed-VC Only: 10
Default Hybrid: 10
Optimized Hybrid: 10

Loading SpeechBrain ECAPA-TDNN...


/usr/local/lib/python3.13/dist-packages/speechbrain/utils/autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for load_if_possible
DEBUG:speechbrain.utils.parameter_transfer:Collecting files (or symlinks) for pretraining in /content/evaluation_models/ecapa_voxceleb.
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Using symlink found at '/content/evaluation_models/ecapa_voxceleb/embedding_model.ckpt'
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["embedding_model"] = /content/evaluation_models/ecapa_voxceleb/embedding_model.ckpt
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Using symlink found at '/content/evaluation_mo

✅ SpeechBrain ECAPA-TDNN loaded

Loading Whisper-small...
The first load may download approximately 1 GB.



/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



✅ CPU EVALUATION MODELS READY
Speaker similarity model: SpeechBrain ECAPA-TDNN
Speech recognition model: Whisper-small
Evaluation device: CPU
Total outputs awaiting evaluation: 40


In [ ]:
# ============================================================
# FINAL FOUR-SYSTEM OBJECTIVE EVALUATION
# 10 unseen samples × 4 systems = 40 outputs
# Metrics:
#   1. Speaker similarity — SpeechBrain ECAPA-TDNN
#   2. WER — Whisper-small + JiWER
#   3. Latency and RTF — saved GPU timing results
# ============================================================

import os
import re
import glob
import string
import unicodedata

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn.functional as F

from jiwer import wer

# ============================================================
# 1. Paths
# ============================================================

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

COMPARISON_DIR = os.path.join(
    BASE_DIR,
    "final_hybrid_comparison"
)

TARGET_PATH = os.path.join(
    BASE_DIR,
    "selected_hybrid_data",
    "kathleen_reference.wav"
)

METADATA_PATH = os.path.join(
    BASE_DIR,
    "final_holdout_data",
    "holdout_metadata.csv"
)

TIMING_PATH = os.path.join(
    COMPARISON_DIR,
    "four_system_timing_sample_metrics.csv"
)

PARTIAL_PATH = os.path.join(
    COMPARISON_DIR,
    "four_system_quality_partial.csv"
)

COMPLETE_PATH = os.path.join(
    COMPARISON_DIR,
    "four_system_complete_evaluation.csv"
)

SUMMARY_PATH = os.path.join(
    COMPARISON_DIR,
    "four_system_objective_summary.csv"
)

# ============================================================
# 2. Define the four evaluated systems
# ============================================================

SYSTEMS = {
    "openvoice_only": {
        "display_name": "OpenVoice Only",
        "parameters": "tau=0.5",
        "files": sorted(glob.glob(
            os.path.join(
                COMPARISON_DIR,
                "openvoice_outputs",
                "tau_0.5",
                "openvoice_sample_*_tau_0.5.wav"
            )
        ))
    },

    "seed_vc_only": {
        "display_name": "Seed-VC Only",
        "parameters": "steps=10",
        "files": sorted(glob.glob(
            os.path.join(
                COMPARISON_DIR,
                "seed_based_outputs",
                "seed_only",
                "seed_only_sample_*_steps_10.wav"
            )
        ))
    },

    "default_hybrid": {
        "display_name": "Default Hybrid",
        "parameters": "tau=0.3, steps=20",
        "files": sorted(glob.glob(
            os.path.join(
                COMPARISON_DIR,
                "seed_based_outputs",
                "default_hybrid",
                "default_hybrid_sample_*_tau_0.3_steps_20.wav"
            )
        ))
    },

    "optimized_hybrid": {
        "display_name": "Optimized Hybrid",
        "parameters": "tau=0.5, steps=10",
        "files": sorted(glob.glob(
            os.path.join(
                COMPARISON_DIR,
                "seed_based_outputs",
                "optimized_hybrid",
                "optimized_hybrid_sample_*_tau_0.5_steps_10.wav"
            )
        ))
    }
}

for system_key, system_info in SYSTEMS.items():
    assert len(system_info["files"]) == 10, (
        f"❌ {system_info['display_name']} has "
        f"{len(system_info['files'])} files instead of 10"
    )

# ============================================================
# 3. Load holdout transcripts
# ============================================================

metadata = pd.read_csv(METADATA_PATH)

assert "sample" in metadata.columns, (
    f"❌ 'sample' column is missing. Columns: {metadata.columns.tolist()}"
)

assert "transcript" in metadata.columns, (
    f"❌ 'transcript' column is missing. Columns: {metadata.columns.tolist()}"
)

metadata["sample"] = pd.to_numeric(
    metadata["sample"],
    errors="raise"
).astype(int)

transcript_map = dict(
    zip(
        metadata["sample"],
        metadata["transcript"].astype(str)
    )
)

assert len(transcript_map) == 10, (
    f"❌ Expected 10 holdout transcripts, found {len(transcript_map)}"
)

# ============================================================
# 4. Helper functions
# ============================================================

def extract_sample_number(audio_path):
    """Extract sample number from every output filename."""

    match = re.search(
        r"sample_(\d+)",
        os.path.basename(audio_path)
    )

    if match is None:
        raise ValueError(
            f"Could not extract sample number from: {audio_path}"
        )

    return int(match.group(1))


def normalize_transcript(text):
    """
    Normalize text before calculating WER:
    lowercase, remove punctuation, and normalize spaces.
    """

    text = unicodedata.normalize("NFKC", str(text))
    text = text.lower()

    text = text.replace("’", "'")
    text = text.replace("–", " ")
    text = text.replace("—", " ")
    text = text.replace("-", " ")

    punctuation_table = str.maketrans(
        "",
        "",
        string.punctuation
    )

    text = text.translate(punctuation_table)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def load_audio_16k(audio_path):
    """Load a mono 16-kHz waveform."""

    waveform, _ = librosa.load(
        audio_path,
        sr=16000,
        mono=True
    )

    waveform = np.asarray(
        waveform,
        dtype=np.float32
    )

    return waveform


@torch.inference_mode()
def extract_speaker_embedding(audio_path):
    """Extract normalized ECAPA-TDNN speaker embedding."""

    waveform = load_audio_16k(audio_path)

    waveform_tensor = torch.from_numpy(
        waveform
    ).unsqueeze(0)

    relative_length = torch.tensor(
        [1.0],
        dtype=torch.float32
    )

    embedding = speaker_model.encode_batch(
        waveform_tensor,
        wav_lens=relative_length,
        normalize=False
    )

    embedding = embedding.squeeze().float()
    embedding = F.normalize(
        embedding,
        p=2,
        dim=0
    )

    return embedding.cpu()


@torch.inference_mode()
def transcribe_audio(audio_path):
    """Transcribe one output using Whisper-small."""

    waveform = load_audio_16k(audio_path)

    model_inputs = whisper_processor(
        waveform,
        sampling_rate=16000,
        return_tensors="pt",
        return_attention_mask=True
    )

    input_features = model_inputs.input_features.to("cpu")

    attention_mask = None
    if hasattr(model_inputs, "attention_mask"):
        attention_mask = model_inputs.attention_mask.to("cpu")

    generation_arguments = {
        "input_features": input_features,
        "language": "en",
        "task": "transcribe",
        "max_new_tokens": 128
    }

    if attention_mask is not None:
        generation_arguments["attention_mask"] = attention_mask

    predicted_ids = whisper_model.generate(
        **generation_arguments
    )

    transcription = whisper_processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0].strip()

    return transcription


# ============================================================
# 5. Extract fixed Kathleen embedding
# ============================================================

print("=" * 90)
print("FINAL FOUR-SYSTEM OBJECTIVE EVALUATION")
print("=" * 90)

print("\nExtracting fixed Kathleen target embedding...")

target_embedding = extract_speaker_embedding(
    TARGET_PATH
)

print("✅ Target embedding extracted")

# ============================================================
# 6. Resume previously completed evaluations if available
# ============================================================

result_columns = [
    "system",
    "display_name",
    "parameters",
    "sample",
    "audio_path",
    "reference_transcript",
    "whisper_transcript",
    "normalized_reference",
    "normalized_whisper",
    "speaker_similarity",
    "wer"
]

if os.path.isfile(PARTIAL_PATH):

    quality_results = pd.read_csv(PARTIAL_PATH)

    required_partial_columns = {
        "system",
        "sample",
        "speaker_similarity",
        "wer"
    }

    if not required_partial_columns.issubset(
        quality_results.columns
    ):
        print(
            "⚠️ Existing partial file has incompatible columns; "
            "starting a new evaluation."
        )

        quality_results = pd.DataFrame(
            columns=result_columns
        )

else:

    quality_results = pd.DataFrame(
        columns=result_columns
    )

# إزالة أي صفوف مكررة
if not quality_results.empty:

    quality_results["sample"] = pd.to_numeric(
        quality_results["sample"],
        errors="coerce"
    ).astype("Int64")

    quality_results = quality_results.drop_duplicates(
        subset=["system", "sample"],
        keep="last"
    )

completed_pairs = set()

for _, completed_row in quality_results.iterrows():

    if pd.notna(completed_row["sample"]):

        completed_pairs.add(
            (
                str(completed_row["system"]),
                int(completed_row["sample"])
            )
        )

print(
    f"\nPreviously completed evaluations: "
    f"{len(completed_pairs)}/40"
)

# ============================================================
# 7. Evaluate all systems
# ============================================================

evaluation_jobs = []

for system_key, system_info in SYSTEMS.items():

    for audio_path in system_info["files"]:

        sample_number = extract_sample_number(
            audio_path
        )

        evaluation_jobs.append({
            "system": system_key,
            "display_name": system_info["display_name"],
            "parameters": system_info["parameters"],
            "sample": sample_number,
            "audio_path": audio_path
        })

evaluation_jobs = sorted(
    evaluation_jobs,
    key=lambda item: (
        list(SYSTEMS.keys()).index(item["system"]),
        item["sample"]
    )
)

for job_number, job in enumerate(
    evaluation_jobs,
    start=1
):

    system_key = job["system"]
    sample_number = job["sample"]
    completion_key = (
        system_key,
        sample_number
    )

    print("\n" + "-" * 90)
    print(
        f"Evaluating {job_number}/40 | "
        f"{job['display_name']} | "
        f"Sample {sample_number:02d}"
    )

    if completion_key in completed_pairs:

        print("✅ Already evaluated — skipped")
        continue

    reference_transcript = transcript_map[
        sample_number
    ]

    # Speaker similarity
    converted_embedding = extract_speaker_embedding(
        job["audio_path"]
    )

    similarity_score = F.cosine_similarity(
        target_embedding,
        converted_embedding,
        dim=0
    ).item()

    # Whisper transcription
    whisper_transcript = transcribe_audio(
        job["audio_path"]
    )

    normalized_reference = normalize_transcript(
        reference_transcript
    )

    normalized_whisper = normalize_transcript(
        whisper_transcript
    )

    # WER
    if normalized_reference:

        wer_score = wer(
            normalized_reference,
            normalized_whisper
        )

    else:
        wer_score = np.nan

    new_result = pd.DataFrame([{
        "system": system_key,
        "display_name": job["display_name"],
        "parameters": job["parameters"],
        "sample": sample_number,
        "audio_path": job["audio_path"],
        "reference_transcript": reference_transcript,
        "whisper_transcript": whisper_transcript,
        "normalized_reference": normalized_reference,
        "normalized_whisper": normalized_whisper,
        "speaker_similarity": similarity_score,
        "wer": wer_score
    }])

    quality_results = pd.concat(
        [quality_results, new_result],
        ignore_index=True
    )

    quality_results = quality_results.drop_duplicates(
        subset=["system", "sample"],
        keep="last"
    )

    quality_results = quality_results.sort_values(
        ["system", "sample"]
    ).reset_index(drop=True)

    # حفظ النتيجة بعد كل ملف
    quality_results.to_csv(
        PARTIAL_PATH,
        index=False
    )

    completed_pairs.add(completion_key)

    print(
        f"Speaker similarity: "
        f"{similarity_score:.4f}"
    )

    print(
        f"WER: "
        f"{wer_score:.4f}"
    )

    print(
        f"Reference: "
        f"{reference_transcript}"
    )

    print(
        f"Whisper: "
        f"{whisper_transcript}"
    )

    print(
        f"✅ Progress saved: "
        f"{len(completed_pairs)}/40"
    )

# ============================================================
# 8. Verify all quality evaluations
# ============================================================

quality_results = quality_results.drop_duplicates(
    subset=["system", "sample"],
    keep="last"
)

quality_results["sample"] = quality_results[
    "sample"
].astype(int)

assert len(quality_results) == 40, (
    f"❌ Evaluation is incomplete: "
    f"{len(quality_results)}/40 results"
)

# ============================================================
# 9. Load and normalize timing metrics
# ============================================================

timing_results = pd.read_csv(
    TIMING_PATH
)

def find_column(dataframe, candidates):

    for candidate in candidates:
        if candidate in dataframe.columns:
            return candidate

    return None


timing_system_column = find_column(
    timing_results,
    [
        "system",
        "system_key",
        "display_name",
        "model",
        "method"
    ]
)

timing_sample_column = find_column(
    timing_results,
    [
        "sample",
        "sample_number",
        "sample_id"
    ]
)

timing_latency_column = find_column(
    timing_results,
    [
        "mean_latency",
        "mean_latency_seconds",
        "latency_seconds",
        "total_latency_seconds"
    ]
)

timing_rtf_column = find_column(
    timing_results,
    [
        "mean_rtf",
        "rtf",
        "mean_total_rtf"
    ]
)

assert timing_system_column is not None, (
    f"❌ Timing system column was not found: "
    f"{timing_results.columns.tolist()}"
)

assert timing_sample_column is not None, (
    f"❌ Timing sample column was not found: "
    f"{timing_results.columns.tolist()}"
)

assert timing_latency_column is not None, (
    f"❌ Timing latency column was not found: "
    f"{timing_results.columns.tolist()}"
)

assert timing_rtf_column is not None, (
    f"❌ Timing RTF column was not found: "
    f"{timing_results.columns.tolist()}"
)


def canonical_system_name(value):

    value = str(value).lower()
    value = value.replace("-", " ")
    value = value.replace("_", " ")

    if "optimized" in value:
        return "optimized_hybrid"

    if "default" in value:
        return "default_hybrid"

    if "openvoice" in value or "open voice" in value:
        return "openvoice_only"

    if "seed" in value:
        return "seed_vc_only"

    return value.strip().replace(" ", "_")


timing_for_merge = pd.DataFrame({
    "system": timing_results[
        timing_system_column
    ].apply(canonical_system_name),

    "sample": pd.to_numeric(
        timing_results[timing_sample_column],
        errors="raise"
    ).astype(int),

    "mean_latency": pd.to_numeric(
        timing_results[timing_latency_column],
        errors="coerce"
    ),

    "mean_rtf": pd.to_numeric(
        timing_results[timing_rtf_column],
        errors="coerce"
    )
})

timing_for_merge = timing_for_merge.drop_duplicates(
    subset=["system", "sample"],
    keep="last"
)

assert len(timing_for_merge) == 40, (
    f"❌ Expected 40 timing rows, found "
    f"{len(timing_for_merge)}"
)

# ============================================================
# 10. Merge quality and timing results
# ============================================================

complete_results = quality_results.merge(
    timing_for_merge,
    on=["system", "sample"],
    how="left",
    validate="one_to_one"
)

assert complete_results["mean_latency"].notna().all(), (
    "❌ Some latency results were not matched"
)

assert complete_results["mean_rtf"].notna().all(), (
    "❌ Some RTF results were not matched"
)

system_order = {
    "openvoice_only": 1,
    "seed_vc_only": 2,
    "default_hybrid": 3,
    "optimized_hybrid": 4
}

complete_results["system_order"] = (
    complete_results["system"]
    .map(system_order)
)

complete_results = complete_results.sort_values(
    ["system_order", "sample"]
).drop(columns=["system_order"])

complete_results.to_csv(
    COMPLETE_PATH,
    index=False
)

# ============================================================
# 11. Create objective summary
# ============================================================

objective_summary = (
    complete_results
    .groupby(
        [
            "system",
            "display_name",
            "parameters"
        ],
        as_index=False
    )
    .agg(
        mean_similarity=(
            "speaker_similarity",
            "mean"
        ),
        std_similarity=(
            "speaker_similarity",
            "std"
        ),
        mean_wer=(
            "wer",
            "mean"
        ),
        std_wer=(
            "wer",
            "std"
        ),
        mean_latency=(
            "mean_latency",
            "mean"
        ),
        std_latency=(
            "mean_latency",
            "std"
        ),
        mean_rtf=(
            "mean_rtf",
            "mean"
        ),
        std_rtf=(
            "mean_rtf",
            "std"
        )
    )
)

objective_summary["system_order"] = (
    objective_summary["system"]
    .map(system_order)
)

objective_summary = (
    objective_summary
    .sort_values("system_order")
    .drop(columns=["system_order"])
    .reset_index(drop=True)
)

numeric_columns = [
    "mean_similarity",
    "std_similarity",
    "mean_wer",
    "std_wer",
    "mean_latency",
    "std_latency",
    "mean_rtf",
    "std_rtf"
]

objective_summary[numeric_columns] = (
    objective_summary[numeric_columns]
    .round(4)
)

objective_summary.to_csv(
    SUMMARY_PATH,
    index=False
)

# ============================================================
# 12. Display final results
# ============================================================

print("\n" + "=" * 100)
print("FINAL FOUR-SYSTEM OBJECTIVE EVALUATION SUMMARY")
print("=" * 100)

print(
    objective_summary[
        [
            "display_name",
            "parameters",
            "mean_similarity",
            "std_similarity",
            "mean_wer",
            "std_wer",
            "mean_latency",
            "std_latency",
            "mean_rtf",
            "std_rtf"
        ]
    ].to_string(index=False)
)

print("\n✅ Evaluated outputs: 40/40")
print("✅ Partial quality results:", PARTIAL_PATH)
print("✅ Complete evaluation:", COMPLETE_PATH)
print("✅ Objective summary:", SUMMARY_PATH)

FINAL FOUR-SYSTEM OBJECTIVE EVALUATION

Extracting fixed Kathleen target embedding...
✅ Target embedding extracted

Previously completed evaluations: 0/40

------------------------------------------------------------------------------------------
Evaluating 1/40 | OpenVoice Only | Sample 01


/tmp/ipykernel_24395/1909851879.py:497: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  quality_results = pd.concat(


Speaker similarity: 0.4077
WER: 0.1667
Reference: We expect that an agent will be available to help you shortly.
Whisper: We expect that inpaging will be available to help you shortly.
✅ Progress saved: 1/40

------------------------------------------------------------------------------------------
Evaluating 2/40 | OpenVoice Only | Sample 02
Speaker similarity: 0.2979
WER: 0.0000
Reference: That's good, otherwise we would have to pay for the bike rack.
Whisper: That's good. Otherwise we would have to pay for the bike rack.
✅ Progress saved: 2/40

------------------------------------------------------------------------------------------
Evaluating 3/40 | OpenVoice Only | Sample 03
Speaker similarity: 0.3180
WER: 0.0000
Reference: It is perfect for families since the kids can safely run around and play in the garden.
Whisper: It is perfect for families since the kids can safely run around and play in the garden.
✅ Progress saved: 3/40

---------------------------------------------------

AssertionError: ❌ Timing RTF column was not found: ['system', 'display_name', 'tau', 'diffusion_steps', 'sample', 'fixed_seed', 'duration_seconds', 'openvoice_latency_seconds', 'mean_seed_latency_seconds', 'std_seed_latency_seconds', 'total_latency_seconds', 'total_rtf', 'output_path']

In [ ]:
# ============================================================
# FIX FINAL MERGE AND CREATE SUMMARY
# Does NOT repeat the 40 audio evaluations
# ============================================================

import os
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"

COMPARISON_DIR = os.path.join(
    BASE_DIR,
    "final_hybrid_comparison"
)

PARTIAL_PATH = os.path.join(
    COMPARISON_DIR,
    "four_system_quality_partial.csv"
)

TIMING_PATH = os.path.join(
    COMPARISON_DIR,
    "four_system_timing_sample_metrics.csv"
)

COMPLETE_PATH = os.path.join(
    COMPARISON_DIR,
    "four_system_complete_evaluation.csv"
)

SUMMARY_PATH = os.path.join(
    COMPARISON_DIR,
    "four_system_objective_summary.csv"
)

# ============================================================
# 1. Load completed quality results
# ============================================================

quality_results = pd.read_csv(PARTIAL_PATH)
timing_results = pd.read_csv(TIMING_PATH)

quality_results["sample"] = pd.to_numeric(
    quality_results["sample"],
    errors="raise"
).astype(int)

quality_results = quality_results.drop_duplicates(
    subset=["system", "sample"],
    keep="last"
)

assert len(quality_results) == 40, (
    f"❌ Quality evaluation incomplete: "
    f"{len(quality_results)}/40"
)

print("✅ Quality evaluations loaded:", len(quality_results))
print("✅ Timing rows loaded:", len(timing_results))

# ============================================================
# 2. Standardize system names
# ============================================================

def canonical_system_name(value):

    value = str(value).lower()
    value = value.replace("-", " ")
    value = value.replace("_", " ")

    if "optimized" in value:
        return "optimized_hybrid"

    if "default" in value:
        return "default_hybrid"

    if "openvoice" in value or "open voice" in value:
        return "openvoice_only"

    if "seed" in value:
        return "seed_vc_only"

    return value.strip().replace(" ", "_")

# ============================================================
# 3. Prepare timing metrics
# Correct columns:
# total_latency_seconds and total_rtf
# ============================================================

timing_for_merge = pd.DataFrame({
    "system": timing_results[
        "system"
    ].apply(canonical_system_name),

    "sample": pd.to_numeric(
        timing_results["sample"],
        errors="raise"
    ).astype(int),

    "mean_latency": pd.to_numeric(
        timing_results["total_latency_seconds"],
        errors="coerce"
    ),

    "mean_rtf": pd.to_numeric(
        timing_results["total_rtf"],
        errors="coerce"
    )
})

timing_for_merge = timing_for_merge.drop_duplicates(
    subset=["system", "sample"],
    keep="last"
)

assert len(timing_for_merge) == 40, (
    f"❌ Expected 40 timing rows, found "
    f"{len(timing_for_merge)}"
)

# ============================================================
# 4. Merge quality and efficiency metrics
# ============================================================

complete_results = quality_results.merge(
    timing_for_merge,
    on=["system", "sample"],
    how="left",
    validate="one_to_one"
)

assert complete_results["mean_latency"].notna().all(), (
    "❌ Some latency values were not matched"
)

assert complete_results["mean_rtf"].notna().all(), (
    "❌ Some RTF values were not matched"
)

system_order = {
    "openvoice_only": 1,
    "seed_vc_only": 2,
    "default_hybrid": 3,
    "optimized_hybrid": 4
}

complete_results["system_order"] = (
    complete_results["system"].map(system_order)
)

complete_results = (
    complete_results
    .sort_values(["system_order", "sample"])
    .drop(columns=["system_order"])
    .reset_index(drop=True)
)

complete_results.to_csv(
    COMPLETE_PATH,
    index=False
)

# ============================================================
# 5. Create final objective summary
# ============================================================

objective_summary = (
    complete_results
    .groupby(
        ["system", "display_name", "parameters"],
        as_index=False
    )
    .agg(
        mean_similarity=("speaker_similarity", "mean"),
        std_similarity=("speaker_similarity", "std"),
        mean_wer=("wer", "mean"),
        std_wer=("wer", "std"),
        mean_latency=("mean_latency", "mean"),
        std_latency=("mean_latency", "std"),
        mean_rtf=("mean_rtf", "mean"),
        std_rtf=("mean_rtf", "std")
    )
)

objective_summary["system_order"] = (
    objective_summary["system"].map(system_order)
)

objective_summary = (
    objective_summary
    .sort_values("system_order")
    .drop(columns=["system_order"])
    .reset_index(drop=True)
)

numeric_columns = [
    "mean_similarity",
    "std_similarity",
    "mean_wer",
    "std_wer",
    "mean_latency",
    "std_latency",
    "mean_rtf",
    "std_rtf"
]

objective_summary[numeric_columns] = (
    objective_summary[numeric_columns].round(4)
)

objective_summary.to_csv(
    SUMMARY_PATH,
    index=False
)

# ============================================================
# 6. Display final table
# ============================================================

print("\n" + "=" * 105)
print("FINAL FOUR-SYSTEM OBJECTIVE EVALUATION SUMMARY")
print("=" * 105)

print(
    objective_summary[
        [
            "display_name",
            "parameters",
            "mean_similarity",
            "std_similarity",
            "mean_wer",
            "std_wer",
            "mean_latency",
            "std_latency",
            "mean_rtf",
            "std_rtf"
        ]
    ].to_string(index=False)
)

print("\n✅ Evaluated outputs: 40/40")
print("✅ Complete evaluation:", COMPLETE_PATH)
print("✅ Objective summary:", SUMMARY_PATH)

✅ Quality evaluations loaded: 40
✅ Timing rows loaded: 40

FINAL FOUR-SYSTEM OBJECTIVE EVALUATION SUMMARY
    display_name        parameters  mean_similarity  std_similarity  mean_wer  std_wer  mean_latency  std_latency  mean_rtf  std_rtf
  OpenVoice Only           tau=0.5           0.3162          0.0643    0.0712   0.1074        0.2622       0.0259    0.0501   0.0047
    Seed-VC Only          steps=10           0.7034          0.0484    0.0487   0.1052        1.3749       0.1995    0.2609   0.0202
  Default Hybrid tau=0.3, steps=20           0.7335          0.0429    0.0830   0.1122        2.2105       0.2986    0.4196   0.0243
Optimized Hybrid tau=0.5, steps=10           0.7203          0.0397    0.0712   0.1074        1.6511       0.2043    0.3139   0.0202

✅ Evaluated outputs: 40/40
✅ Complete evaluation: /content/drive/MyDrive/IJEEE_VC_data/final_hybrid_comparison/four_system_complete_evaluation.csv
✅ Objective summary: /content/drive/MyDrive/IJEEE_VC_data/final_hybrid_comparison